In [2]:
import requests
import json

In [84]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        print(results)
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']
            state = results['result']['geographies']['2020 Census Blocks'][0]['STATE']
            print(state)
            
            return tract, county
        except IndexError:
            print("[ERROR] Unable to retrieve census geography for: " + location)
        except KeyError:
            print("[ERROR] Location is outside of the United States: " + location)
        except Exception as error:
            print(f"[ERROR] Error retrieving census geography for: {location}, Error: {error}")

    print("[ERROR] API call failed for: " + location + " with coordinates" + str(coordinates))
    return None, None  # Return this if API call failed or no tracts found

In [85]:
result = query_census_api("department of public health", [-71.057716, 42.35807])
tract, county = result

{'result': {'geographies': {'2020 Census Blocks': [{'SUFFIX': '', 'GEOID': '250250303021041', 'CENTLAT': '+42.3582339', 'BLOCK': '1041', 'AREAWATER': 0, 'STATE': '25', 'BASENAME': '1041', 'OID': '210701007749506', 'LSADC': 'BK', 'FUNCSTAT': 'S', 'INTPTLAT': '+42.3582339', 'NAME': 'Block 1041', 'OBJECTID': 7984851, 'TRACT': '030302', 'CENTLON': '-071.0575616', 'BLKGRP': '1', 'AREALAND': 7292, 'INTPTLON': '-071.0575616', 'MTFCC': 'G5040', 'LWBLKTYP': 'L', 'UR': 'U', 'COUNTY': '025'}]}, 'input': {'vintage': {'isDefault': True, 'id': '4', 'vintageName': 'Current_Current', 'vintageDescription': 'Current Vintage - Current Benchmark'}, 'location': {'x': -71.057716, 'y': 42.35807}, 'benchmark': {'isDefault': True, 'benchmarkDescription': 'Public Address Ranges - Current Benchmark', 'id': '4', 'benchmarkName': 'Public_AR_Current'}}}}
25


In [76]:
def query_census_reporter(tract, state, county):
    geoid = f'14000US{state}{county}{tract}'

    url = f"https://api.censusreporter.org/1.0/geo/latest/{geoid}"

    response = requests.get(url)
    print(response)
    # Check if the response is successful
    if response.status_code == 200:
        data = response.json()
        print(data)

        # Attempt to extract the place (city) from the response
        try:
            place_name = data['geography']['name']
            return place_name
        except KeyError:
            print(f"[ERROR] Place not found for GEOID: {geoid}")
            return None
    else:
        print(f"[ERROR] Failed to retrieve data from Census Reporter API for GEOID: {geoid}")
        return None

In [77]:
city = query_census_reporter(tract, '25', county)

<Response [200]>
{'geometry': None, 'properties': {'aland': '477388', 'awater': '0', 'display_name': 'Census Tract 303.02, Suffolk, MA', 'full_geoid': '14000US25025030302', 'population': 2399, 'simple_name': 'Census Tract 303.02', 'sumlevel': '140'}, 'type': 'Feature'}
[ERROR] Place not found for GEOID: 14000US25025030302


In [45]:
# Get census demographics for any given article
def get_census_demographics(year, dsource, dname, tract, county, state):
    cols = 'NAME,PLACE,PLACEREM,STATE,CONCIT,REGION'
    base_url = f"https://api.census.gov/data/{year}/{dsource}/{dname}"

    census_url = f"{base_url}?get={cols}&for=tract:{tract}&in=county:{county}&in=state:{state}"

    census_response = requests.get(census_url)
    census_response_json = census_response.json()

    return census_response_json

In [47]:
state = '25'

In [48]:
census_data = get_census_demographics("2020", "dec", "pl", tract, county, state)

In [49]:
census_data

[['NAME',
  'PLACE',
  'PLACEREM',
  'STATE',
  'CONCIT',
  'REGION',
  'state',
  'county',
  'tract'],
 ['Census Tract 303.02, Suffolk County, Massachusetts',
  None,
  None,
  '25',
  None,
  None,
  '25',
  '025',
  '030302']]

In [78]:
import googlemaps
from dotenv import load_dotenv
import os

In [79]:
# Load environment variables
load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [80]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [82]:
ma_center_coords = (42.4072, -71.3824) 

# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']

            print(geocode_result)
            coords = [latitude, longitude]
            
            return longitude, latitude
        else:
            print(f"[WARNING] Could not find location {location} through google maps")
            return None, None
    except Exception as error:
        print(f"[ERROR] Error finding locations through google maps, {error}")
        return None, None

In [83]:
gmaps_results = callGoogleMapsAPI("department of public health")

[{'address_components': [{'long_name': '250', 'short_name': '250', 'types': ['street_number']}, {'long_name': 'Washington Street', 'short_name': 'Washington St', 'types': ['route']}, {'long_name': 'Downtown', 'short_name': 'Downtown', 'types': ['neighborhood', 'political']}, {'long_name': 'Boston', 'short_name': 'Boston', 'types': ['locality', 'political']}, {'long_name': 'Suffolk County', 'short_name': 'Suffolk County', 'types': ['administrative_area_level_2', 'political']}, {'long_name': 'Massachusetts', 'short_name': 'MA', 'types': ['administrative_area_level_1', 'political']}, {'long_name': 'United States', 'short_name': 'US', 'types': ['country', 'political']}, {'long_name': '02108', 'short_name': '02108', 'types': ['postal_code']}], 'formatted_address': '250 Washington St, Boston, MA 02108, USA', 'geometry': {'location': {'lat': 42.35807, 'lng': -71.057716}, 'location_type': 'ROOFTOP', 'viewport': {'northeast': {'lat': 42.35951903029149, 'lng': -71.0563713697085}, 'southwest': {'

(-71.057716, 42.35807)

In [86]:
city = "Boston"

In [93]:
def get_city_demographics(year, dsource, dname, city, state):
    cols = 'NAME,P2_001N,P2_002N,P2_003N,P2_004N,P2_005N,P2_006N,P2_007N,P2_008N,P2_009N,P2_010N'
    base_url = f"https://api.census.gov/data/{year}/{dsource}/{dname}"

    # Note: Adjust 'for' and 'in' parameters based on city-level geography
    census_url = f"{base_url}?get={cols}&for=place:*&in=state:{state}"

    census_response = requests.get(census_url)
    print(census_response)
    census_response_json = census_response.json()

    print(census_response_json)
    
    # Filter results to find the specific city
    city_demographics = [
        item for item in census_response_json[1:]
        if city in item[0]  # Assuming the city name is in the first column of the results
    ]

    return city_demographics


In [94]:
city_demographics = get_city_demographics("2020", "dec", "pl", city, state)

<Response [200]>
[['NAME', 'P2_001N', 'P2_002N', 'P2_003N', 'P2_004N', 'P2_005N', 'P2_006N', 'P2_007N', 'P2_008N', 'P2_009N', 'P2_010N', 'state', 'place'], ['Abington CDP, Massachusetts', '17062', '660', '16402', '15522', '14018', '661', '26', '425', '9', '383', '25', '00135'], ['Acushnet Center CDP, Massachusetts', '3030', '66', '2964', '2856', '2797', '15', '9', '13', '0', '22', '25', '00530'], ['Adams CDP, Massachusetts', '5466', '170', '5296', '5020', '4904', '75', '5', '21', '0', '15', '25', '00590'], ['Agawam Town city, Massachusetts', '28692', '1827', '26865', '25941', '24567', '529', '28', '727', '3', '87', '25', '00840'], ['Amesbury Town city, Massachusetts', '17366', '675', '16691', '16065', '15572', '215', '32', '167', '5', '74', '25', '01260'], ['Amherst Town city, Massachusetts', '39263', '3807', '35456', '33614', '24135', '2228', '59', '6975', '14', '203', '25', '01370'], ['Andover CDP, Massachusetts', '9735', '677', '9058', '8727', '7696', '218', '2', '768', '1', '42', '

In [95]:
print(city_demographics)

[['Boston city, Massachusetts', '675647', '126113', '549534', '516813', '301464', '129264', '989', '75588', '251', '9257', '25', '07000']]


In [109]:
# Get census demographics for any given article
def get_census_demographics(year, dsource, dname, tract, county, state):
    cols = 'NAME,P2_001N,P2_002N,P2_003N,P2_004N,P2_005N,P2_006N,P2_007N,P2_008N,P2_009N,P2_010N'
    base_url = f"https://api.census.gov/data/{year}/{dsource}/{dname}"

    census_url = f"{base_url}?get={cols}&for=tract:{tract}&in=county:{county}&in=state:{state}"

    census_response = requests.get(census_url)
    census_response_json = census_response.json()

    return census_response_json

def get_city_demographics(year, dsource, dname, city, state):
    cols = 'NAME,P2_001N,P2_002N,P2_003N,P2_004N,P2_005N,P2_006N,P2_007N,P2_008N,P2_009N,P2_010N'
    base_url = f"https://api.census.gov/data/{year}/{dsource}/{dname}"

    # Note: Adjust 'for' and 'in' parameters based on city-level geography
    census_url = f"{base_url}?get={cols}&for=place:*&in=state:{state}"

    census_response = requests.get(census_url)
    census_response_json = census_response.json()
    
    # Filter results to find the specific city
    city_demographics = [
        item for item in census_response_json[1:]
        if city in item[0]  # Assuming the city name is in the first column of the results
    ]
    print(census_response_json)
    print(city_demographics)
    columns = cols.split(",")
    city_demographics = [columns, city_demographics[0]]
    return city_demographics



In [110]:
tract = "215101"
county = "009"
city = "Hamilton"
state = "25"

tract_census_data = get_census_demographics("2020", "dec", "pl", tract, county, state)
city_census_data = get_city_demographics("2020", "dec", "pl", city, state)
print(tract_census_data)
print(city_census_data)


[['NAME', 'P2_001N', 'P2_002N', 'P2_003N', 'P2_004N', 'P2_005N', 'P2_006N', 'P2_007N', 'P2_008N', 'P2_009N', 'P2_010N', 'state', 'place'], ['Abington CDP, Massachusetts', '17062', '660', '16402', '15522', '14018', '661', '26', '425', '9', '383', '25', '00135'], ['Acushnet Center CDP, Massachusetts', '3030', '66', '2964', '2856', '2797', '15', '9', '13', '0', '22', '25', '00530'], ['Adams CDP, Massachusetts', '5466', '170', '5296', '5020', '4904', '75', '5', '21', '0', '15', '25', '00590'], ['Agawam Town city, Massachusetts', '28692', '1827', '26865', '25941', '24567', '529', '28', '727', '3', '87', '25', '00840'], ['Amesbury Town city, Massachusetts', '17366', '675', '16691', '16065', '15572', '215', '32', '167', '5', '74', '25', '01260'], ['Amherst Town city, Massachusetts', '39263', '3807', '35456', '33614', '24135', '2228', '59', '6975', '14', '203', '25', '01370'], ['Andover CDP, Massachusetts', '9735', '677', '9058', '8727', '7696', '218', '2', '768', '1', '42', '25', '01430'], ['

IndexError: list index out of range

In [ ]:
def update_demographics(tract, county, state, city=None):
    try:
        if city:
            census_data = get_city_demographics("2020", "dec", "pl", city, state)
        else:
            census_data = get_census_demographics("2020", "dec", "pl", tract, county, state)
        
        if not census_data or len(census_data) < 2:
            raise ValueError("Census data is missing or malformed.")
        
        headers = census_data[0]  # Headers
        values = census_data[1]   # Data values
        data = dict(zip(headers, values))

        county_name = data.get('NAME', "")
        geoid_tract = f"{state}{county}{tract}"

        # Prepare the update document
        update_doc = {
            'demographics.p2_001n': str(data.get('P2_001N', 0)),
            'demographics.p2_002n': str(data.get('P2_002N', 0)),
            'demographics.p2_003n': str(data.get('P2_003N', 0)),
            'demographics.p2_004n': str(data.get('P2_004N', 0)),
            'demographics.p2_005n': str(data.get('P2_005N', 0)),
            'demographics.p2_006n': str(data.get('P2_006N', 0)),
            'demographics.p2_007n': str(data.get('P2_007N', 0)),
            'demographics.p2_008n': str(data.get('P2_008N', 0)),
            'demographics.p2_009n': str(data.get('P2_009N', 0)),
            'demographics.p2_010n': str(data.get('P2_010N', 0)),
            'county_name': county_name,
            'geoid_tract': geoid_tract
        }
        
        print(update_doc)
        print(f"Tract {tract} updated with census data.")

    except Exception as error:
        print(f"[ERROR] Error getting census data for tract {tract}: {error}")
        
        return